In [ ]:
import torch
from ultralytics import YOLO

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

# --- Veri seti yolu ---
# Once depo icindeki dataset/ klasoruna bakar (README'deki kurulum),
# bulamazsa gelistirme makinesindeki konuma duser.
import os

_CANDIDATES = [
    "../dataset/data.yaml",
    "../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
]
DATA = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[0])
print("veri seti:", os.path.abspath(DATA), "->", os.path.exists(DATA))

In [ ]:
model = YOLO("yolo26n.pt")

results = model.train(
        data=DATA,       # dataset yaml dosyasi
        epochs=500,              # toplam epoch sayisi (dataset buyuklugune gore ayarla)
        imgsz=640,                # girdi resim boyutu
        batch=16,                  # batch size (GPU bellegine gore dusur/yukselt: 8, 16, 32...)
        device=device,         # 0 -> ilk GPU, "cpu" -> CPU
        #workers=4,               # veri yukleme islemci sayisi
        plots=True,                           # egitim grafiklerini olustur (loss, mAP, vb.)
        verbose=True,
    )

In [ ]:
######################################## Test Kodları ##############################################
from ultralytics import YOLO
from pathlib import Path
import torch

model_path = Path("runs/detect/train/weights/best.pt")
if model_path.exists():
    model = YOLO(str(model_path))
    print(f"Test ediliyor: {model_path}")
    results = model.val(
        data=DATA,
        split="test",
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
    )

    metrics = results.results_dict
    precision = metrics.get("metrics/precision(B)", None)
    recall = metrics.get("metrics/recall(B)", None)
    map50 = metrics.get("metrics/mAP50(B)", None)
    map5095 = metrics.get("metrics/mAP50-95(B)", None)

    print("\nTest metrikleri:")
    print(f"Precision: {precision * 100:.2f}%" if precision is not None else "Precision: bulunamadı")
    print(f"Recall: {recall * 100:.2f}%" if recall is not None else "Recall: bulunamadı")
    print(f"mAP@0.5: {map50 * 100:.2f}%" if map50 is not None else "mAP@0.5: bulunamadı")
    print(f"mAP@0.5-0.95: {map5095 * 100:.2f}%" if map5095 is not None else "mAP@0.5-0.95: bulunamadı")
else:
    print(f"Model bulunamadı: {model_path}")
